In [24]:
import sys
import os
sys.path.append("..")

In [25]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.neural_network import MLPClassifier

from bonXAI.core.preprocessing import Preprocessor
from bonXAI.core.explainer import Explainer
from bonXAI.core.evaluation import Evaluator
from bonXAI.core.utils import set_global_seed, possible_g_values, possible_num_bins_values

# from openxai.model import LoadModel

SEED = 42
set_global_seed(SEED)

In [39]:
# generate data
X, y = make_classification(n_samples=20, n_features=7, n_classes=2, random_state=SEED)
df_data = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df_data["label"] = y

# or load data
# _, loader_test = ReturnLoaders(data_name='german', download=False, batch_size=128)
# X_test = loader_test.dataset.data
# y_test = loader_test.dataset.targets.to_numpy()

In [40]:
# train model
model = MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=SEED)
model.fit(X, y)

# or load model
# model = LoadModel(data_name='german', ml_model='ann', pretrained=True)
# model.eval()

MLPClassifier(hidden_layer_sizes=(10,), max_iter=500, random_state=42)

In [41]:
# calculate Ground Truth
gt_explainer = Explainer(model=model, name="shap", variant="kernel", seed=SEED)
exp_gt, t_gt = gt_explainer.explain(X, y)
exp_gt_mean = exp_gt.mean(axis=0)

# or load GT
# shap_values_gt = ...

In [ ]:
# save GT explanation
data_name = "test_data" 
num_repeats = 1
save_dir = f"metadata/{data_name}"
os.makedirs(save_dir, exist_ok=True)
npz_path = os.path.join(save_dir, f"ground_truth_shap_kernel_{num_repeats}.npz")
np.savez_compressed(npz_path, explanation_values=exp_gt, explanation_mean=exp_gt_mean, time=t_gt)
print(f"Saved explanation to {npz_path}")
csv_path = npz_path.replace(".npz", ".csv")
df = pd.DataFrame(exp_gt_mean.reshape(1, -1))
df.to_csv(csv_path, index=False)
print(f"Saved mean explanation to {csv_path}")

In [42]:
results = []

methods = ["iid", "compress", "compress_with_predictions", "compress_stratified"]
# kernels = ["gaussian", "sobolev", "inverse_multiquadric"]
kernels = ["gaussian", "sobolev"]
n_samples = X.shape[0]
bins_list = possible_num_bins_values(n_samples)

evaluator = Evaluator(ground_truth_explanation=exp_gt_mean, reference_points=X)

for method in methods:
    if method == "iid":
            pre = Preprocessor(method=method, model=model, kernel="gaussian", seed=SEED)
            X_red, y_red, idx, t_comp = pre.run(X, y)
            print(f"{method} compression complete")
            for explainer_name, variant in [("shap", "kernel"), ("sage", "permutation")]:
                explainer = Explainer(model=model, name=explainer_name, variant=variant, seed=SEED)
                values, t_exp = explainer.explain(X_red, y_red)
                exp_mean = values
                if exp_mean.ndim > 1:
                    exp_mean = values.mean(axis=0)
                row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
                row.update(evaluator.evaluate_compression(X_red))
                row.update({
                    "method": method,
                    "g": None,
                    "num_bins": None,
                    "compression_time": t_comp,
                    "kernel": None,
                    "explainer": explainer_name,
                    "variant": variant,
                })
                results.append(row)
                print(f"{method} {explainer_name} {variant} explanation complete")
    else:
        for kernel in kernels:
            for num_bins in bins_list:
                g_list = possible_g_values(n_samples, num_bins)
                for g in g_list:
                    pre = Preprocessor(method=method, model=model, g=g, num_bins=num_bins, kernel=kernel, seed=SEED)
                    X_red, y_red, idx, t_comp = pre.run(X, y)
                    print(f"{method} compression with g={g}, num_bins={num_bins}, kernel={kernel} complete")
                    for explainer_name, variant in [("shap", "kernel"), ("sage", "permutation")]:
                        explainer = Explainer(model=model, name=explainer_name, variant=variant, seed=SEED)
                        values, t_exp = explainer.explain(X_red, y_red)
                        exp_mean = values
                        if exp_mean.ndim > 1:
                            exp_mean = values.mean(axis=0)
                        row = evaluator.evaluate_explanation(exp_mean, t_exp, len(X_red))
                        row.update(evaluator.evaluate_compression(X_red))
                        row.update({
                            "method": method,
                            "g": g,
                            "num_bins": num_bins,
                            "compression_time": t_comp,
                            "kernel": kernel,
                            "explainer": explainer_name,
                            "variant": variant,
                        })
                        results.append(row)
                        print(f"{method} {explainer_name} {variant} with g={g}, num_bins={num_bins}, kernel={kernel} explanation complete")


iid compression complete
iid shap kernel explanation complete
PermutationEstimator will use 16 jobs
iid sage permutation explanation complete
compress compression with g=2, num_bins=1, kernel=gaussian complete
compress shap kernel with g=2, num_bins=1, kernel=gaussian explanation complete
PermutationEstimator will use 16 jobs
compress sage permutation with g=2, num_bins=1, kernel=gaussian explanation complete
compress compression with g=1, num_bins=1, kernel=gaussian complete
compress shap kernel with g=1, num_bins=1, kernel=gaussian explanation complete
PermutationEstimator will use 16 jobs
compress sage permutation with g=1, num_bins=1, kernel=gaussian explanation complete
compress compression with g=0, num_bins=1, kernel=gaussian complete
compress shap kernel with g=0, num_bins=1, kernel=gaussian explanation complete
PermutationEstimator will use 16 jobs
compress sage permutation with g=0, num_bins=1, kernel=gaussian explanation complete
compress compression with g=2, num_bins=1, ke

In [43]:
df_results = pd.DataFrame(results)
df_results.sort_values(by="mae")
df_results

,mae,top_k,time,size,mmd,method,g,num_bins,compression_time,kernel,explainer,variant
0,1.807830e-17,0.8,0.014236,4,0.138704,iid,NaN,NaN,0.000039,None,shap,kernel
1,6.474704e-02,0.8,0.042086,4,0.138704,iid,NaN,NaN,0.000039,None,sage,permutation
2,2.012899e-17,1.0,0.127928,16,0.009028,compress,2.0,1.0,0.000096,gaussian,shap,kernel
3,7.769641e-02,1.0,0.280553,16,0.009028,compress,2.0,1.0,0.000096,gaussian,sage,permutation
4,1.062518e-17,0.6,0.022235,8,0.027799,compress,1.0,1.0,0.000075,gaussian,shap,kernel
5,7.641228e-02,0.8,0.236236,8,0.027799,compress,1.0,1.0,0.000075,gaussian,sage,permutation
6,2.012899e-17,1.0,0.099160,16,0.009028,compress,0.0,1.0,0.000096,gaussian,shap,kernel
7,7.769641e-02,1.0,0.270364,16,0.009028,compress,0.0,1.0,0.000096,gaussian,sage,permutation
8,2.012899e-17,1.0,0.119848,16,0.009028,compress,2.0,1.0,0.000197,sobolev,shap,kernel
9,7.769641e-02,1.0,0.230014,16,0.009028,compress,2.0,1.0,0.000197,sobolev,sage,permutation


In [ ]:
# data_name = "test_data" 
# num_repeats = 1
# save_dir = f"metadata/{data_name}"
# os.makedirs(save_dir, exist_ok=True)

csv_path = os.path.join(save_dir, f"experiment_results_{num_repeats}.csv")
df_results.to_csv(csv_path, index=False)
print(f"Saved DataFrame to CSV: {csv_path}")